# Machine-learning workflow template: one extent, one predictor set

This notebook shows the seabed-litter suitability workflow for **one** model set: the **full extent** with
**all predictors** (V1). The other extents and predictor groups use the same code with a different `EXTENT` / `VARIANT`.

Steps:
1. Load the predictor rasters and the MEDITS presences
2. Predictor screening (coverage, correlation, VIF)
3. Pseudo-absences (5 km buffer + environmental envelope)
4. Spatial split (micro-clusters, 1 holdout group + 5 CV folds)
5. Train RF, XGBoost, SVM, MaxEnt and MLP (default vs tuned)
6. Weighted ensemble, thresholds and holdout evaluation
7. Permutation importance and SHAP
8. Prediction map
9. Compact model (backward elimination)

The predictor rasters are already on the common 1/120° grid (one GeoTIFF per predictor).

## 0. Imports and settings

In [ ]:
import gc
import json
from itertools import combinations, product
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.windows import Window
import matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial import cKDTree
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedGroupKFold, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve, confusion_matrix,
                             f1_score, cohen_kappa_score)
from xgboost import XGBClassifier
import elapid as ela
import shap

In [ ]:
EXTENT = "Full_extent"          # "Full_extent" or "Malta_25nm"
VARIANT = "V1_All_Data"         # V1_All_Data, V2_Env, V3_Human, V4_Oceanic
VARIANT_GROUPS = {"V1_All_Data": ["environmental", "oceanic", "human"], "V2_Env": ["environmental"],
                  "V3_Human": ["human"], "V4_Oceanic": ["oceanic"]}

DATA_DIR = Path("data/ml")
PREDICTOR_DIR = DATA_DIR / "aligned_predictors" / EXTENT      # one GeoTIFF per predictor, same grid
MANIFEST = DATA_DIR / "predictor_manifest.csv"               # columns: predictor, group (environmental/oceanic/human)
PRESENCE_FILE = DATA_DIR / "medits_haul_midpoints.geojson"   # MEDITS haul midpoints (EMODnet)
OUT_DIR = Path("outputs/ml") / f"{EXTENT}_{VARIANT}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 20260916
METRIC_CRS = "EPSG:3035"

# screening
MIN_COVERAGE, MIN_UNIQUE = 0.90, 5
CORR_FLAG, VIF_MAX = 0.70, 10.0
SUBSTRATE_REFERENCE = "substrate_fine_or_sandy_or_muddy"     # most abundant class, dropped as the reference
LOG1P_PREFIXES = ("vessel_density_", "route_density_", "kinetic_energy_")

# pseudo-absences
BUFFER_KM = 5.0
SRE_PERCENTILES = (2.5, 97.5)
N_PSEUDO_ABSENCES = 1000
N_EXTRA_DRAWS = 5               # only used to check the VIF screening is stable

# spatial split and models
N_GROUPS, N_FOLDS, HOLDOUT_GROUP = 6, 5, 0
MODEL_NAMES = ["Random_Forest", "XGBoost", "SVM", "MaxEnt", "MLP"]
N_BOOTSTRAP, PERM_REPEATS = 2000, 5
SHAP_SAMPLES, SHAP_BACKGROUND = 100, 50

## 1. Predictors and presences

In [ ]:
manifest = pd.read_csv(MANIFEST)
group_of = dict(zip(manifest.predictor, manifest.group))

with rasterio.open(PREDICTOR_DIR / "bathymetry.tif") as src:
    sea = src.read(1, masked=True).filled(np.nan) < 0       # marine cells = below sea level
    transform, crs, shape = src.transform, src.crs, src.shape

rows, cols = np.where(sea)
grid = pd.DataFrame({"row": rows, "col": cols,
                     "lon": transform.c + (cols + 0.5) * transform.a,
                     "lat": transform.f + (rows + 0.5) * transform.e})
for name in manifest.predictor:
    with rasterio.open(PREDICTOR_DIR / f"{name}.tif") as src:
        grid[name] = src.read(1, masked=True).filled(np.nan)[sea]
print("Marine cells:", len(grid), "| predictors:", len(manifest))

In [ ]:
# each haul midpoint -> grid cell; hauls in the same cell become one presence
hauls = gpd.read_file(PRESENCE_FILE).to_crs("EPSG:4326")
c, r = ~transform * (hauls.geometry.x.values, hauls.geometry.y.values)
r, c = np.floor(r).astype(int), np.floor(c).astype(int)
lookup = {(a, b): i for i, (a, b) in enumerate(zip(grid.row, grid.col))}
cell_idx = np.array([lookup[(a, b)] for a, b in zip(r, c) if (a, b) in lookup])
presence_cells = np.unique(cell_idx)
print(f"Hauls: {len(hauls)}, on marine cells: {len(cell_idx)}, presences after merging by cell: {len(presence_cells)}")

## 2. Predictor screening

Screening is done on **all marine cells**, so it describes the area that will be predicted (not the presences).
Predictors with < 90% coverage or < 5 distinct values are removed, identical layers are dropped, the reference
substrate class is removed (the fractions add up to 1), skewed density layers are log-transformed, and then predictors are
removed one at a time while any VIF is above 10.

In [ ]:
candidates = [p for p in manifest.predictor if group_of[p] in VARIANT_GROUPS[VARIANT]]

coverage = grid[candidates].notna().mean()
n_unique = grid[candidates].nunique()
keep = [p for p in candidates if coverage[p] >= MIN_COVERAGE and n_unique[p] >= MIN_UNIQUE]

# identical layers (e.g. some distance-to-port layers were exactly the same)
dupes = set()
for a, b in combinations(keep, 2):
    if b not in dupes and np.allclose(grid[a], grid[b], equal_nan=True):
        dupes.add(b)
keep = [p for p in keep if p not in dupes and p != SUBSTRATE_REFERENCE]

def transform_table(frame, columns):
    out = frame[columns].copy()
    for col in columns:
        if col.startswith(LOG1P_PREFIXES):
            out[col] = np.log1p(out[col])
    return out

X_grid = transform_table(grid, keep)
print("After coverage / duplicates / reference class:", len(keep))
print("Dropped duplicates:", sorted(dupes))

In [ ]:
def vif(frame, columns):
    X = frame[columns].dropna().to_numpy(float)
    Z = (X - X.mean(0)) / X.std(0)
    out = {}
    for i, name in enumerate(columns):
        others = np.column_stack([np.ones(len(Z)), np.delete(Z, i, axis=1)])
        coef, *_ = np.linalg.lstsq(others, Z[:, i], rcond=None)
        resid = Z[:, i] - others @ coef
        r2 = 1 - resid @ resid / ((Z[:, i] - Z[:, i].mean()) ** 2).sum()
        out[name] = np.inf if r2 >= 1 else 1 / (1 - r2)
    return pd.Series(out).sort_values(ascending=False)


def iterative_vif(frame, columns, threshold=VIF_MAX):
    kept, removed = list(columns), []
    while True:
        v = vif(frame, kept)
        if v.iloc[0] <= threshold:
            return kept, removed
        removed.append((v.index[0], round(float(v.iloc[0]), 1)))
        kept.remove(v.index[0])

corr = X_grid.corr().abs()
flagged = [(a, b, round(corr.loc[a, b], 2)) for a, b in combinations(keep, 2) if corr.loc[a, b] > CORR_FLAG]
print(f"Pairs with |r| > {CORR_FLAG}: {len(flagged)}")

FEATURES, vif_removed = iterative_vif(X_grid, keep)
print("Removed for VIF:", vif_removed)
print("Retained:", len(FEATURES))

## 3. Pseudo-absences

MEDITS only records where litter was found, so pseudo-absences are needed. A cell is eligible if it is more than 5 km
from a presence (about one haul length) **and** at least one environmental or oceanographic predictor is outside the
2.5–97.5 percentile range seen at the presences. Human predictors are not used here so they can't influence the selection.
1000 are drawn at random with a fixed seed.

In [ ]:
xy = gpd.GeoSeries(gpd.points_from_xy(grid.lon, grid.lat), crs="EPSG:4326").to_crs(METRIC_CRS)
xy = np.column_stack([xy.x, xy.y])
dist_km = cKDTree(xy[presence_cells]).query(xy)[0] / 1000
beyond_buffer = dist_km > BUFFER_KM

# envelope from the environmental + oceanographic predictors of the all-predictor set
# (the same pseudo-absences are shared by all four predictor sets, so this always uses the V1 predictors)
env_cols = [p for p in FEATURES if group_of[p] in ("environmental", "oceanic")]
substrate = [p for p in FEATURES if p.startswith("substrate_")]
if substrate:   # put the reference substrate class back for the envelope
    grid[SUBSTRATE_REFERENCE] = 1 - grid[substrate].sum(axis=1)
    env_cols.append(SUBSTRATE_REFERENCE)
pres_vals = grid.loc[presence_cells, env_cols]
low, high = pres_vals.quantile(SRE_PERCENTILES[0] / 100), pres_vals.quantile(SRE_PERCENTILES[1] / 100)
inside_envelope = ((grid[env_cols] >= low) & (grid[env_cols] <= high)).all(axis=1).to_numpy()

complete = grid[FEATURES].notna().all(axis=1).to_numpy()
eligible = beyond_buffer & ~inside_envelope & complete
eligible[presence_cells] = False
pool = np.where(eligible)[0]
print(f"Eligible cells: {len(pool):,}")


def make_table(seed):
    absences = np.random.default_rng(seed).choice(pool, size=N_PSEUDO_ABSENCES, replace=False)
    cells = np.concatenate([presence_cells, absences])
    table = transform_table(grid.iloc[cells], FEATURES).reset_index(drop=True)
    table.insert(0, "presence", np.r_[np.ones(len(presence_cells), int), np.zeros(len(absences), int)])
    table["lon"], table["lat"] = grid.lon.values[cells], grid.lat.values[cells]
    return table

data = make_table(SEED)          # the one table used for all modelling
print(data.presence.value_counts())

In [ ]:
# VIF re-checked on the modelling table and 5 extra draws, keep only predictors that pass every time
removed_any = set()
for k in range(N_EXTRA_DRAWS + 1):
    _, removed = iterative_vif(make_table(SEED + k), FEATURES)
    removed_any |= {name for name, _ in removed}
FEATURES = [f for f in FEATURES if f not in removed_any]
print("Removed in at least one draw:", sorted(removed_any))
print("Final predictors:", len(FEATURES))
json.dump(FEATURES, open(OUT_DIR / "final_predictors.json", "w"), indent=2)

## 4. Spatial split

Points are grouped into many small K-means clusters (on projected coordinates), and the clusters are combined into
6 groups of similar size and presence share with `StratifiedGroupKFold`. Group 0 is the independent holdout and the
other 5 are the cross-validation folds.

In [ ]:
pts = gpd.GeoSeries(gpd.points_from_xy(data.lon, data.lat), crs="EPSG:4326").to_crs(METRIC_CRS)
coords = StandardScaler().fit_transform(np.column_stack([pts.x, pts.y]))
y = data.presence.to_numpy()


def stratified_groups(clusters, seed, tries=8):
    """Combine clusters into N_GROUPS groups; keep the most balanced attempt."""
    best, best_score = None, np.inf
    for a in range(tries):
        sgkf = StratifiedGroupKFold(n_splits=N_GROUPS, shuffle=True, random_state=seed + a)
        labels = np.full(len(y), -1)
        for g, (_, test) in enumerate(sgkf.split(np.zeros((len(y), 1)), y, clusters)):
            labels[test] = g
        if any(y[labels == g].sum() == 0 for g in range(N_GROUPS)):
            continue
        size = sum(((labels == g).sum() / (len(y) / N_GROUPS) - 1) ** 2 for g in range(N_GROUPS))
        prev = sum((y[labels == g].sum() / (y.sum() / N_GROUPS) - 1) ** 2 for g in range(N_GROUPS))
        if size + prev < best_score:
            best, best_score = labels, size + prev
    return best, best_score

best = (None, np.inf, None)
for mult in [6, 8, 10, 12, 15, 20, 30, 40, 60, 80]:          # try different numbers of micro-clusters
    k = min(len(y), N_GROUPS * mult)
    for a in range(8):
        clusters = KMeans(n_clusters=k, random_state=SEED + a, n_init=10).fit_predict(coords)
        labels, score = stratified_groups(clusters, SEED + a)
        if labels is not None and score < best[1]:
            best = (labels, score, k)
data["group"] = best[0]
print(f"{best[2]} micro-clusters, balance score {best[1]:.4f}")
print(data.groupby("group").presence.agg(["size", "sum"]))

In [ ]:
dev = data[data.group != HOLDOUT_GROUP].reset_index(drop=True)
hold = data[data.group == HOLDOUT_GROUP].reset_index(drop=True)
dev["fold"] = dev.group.map({g: i for i, g in enumerate(sorted(dev.group.unique()))})

X_dev, y_dev, folds = dev[FEATURES], dev.presence, dev.fold
X_hold, y_hold = hold[FEATURES], hold.presence
print(f"Development: {len(dev)} ({y_dev.sum()} presences) | Holdout: {len(hold)} ({y_hold.sum()} presences)")

## 5. Models

Class imbalance is handled inside each training fold (balanced weights for RF and SVM, `scale_pos_weight` for XGBoost,
oversampling presences for the MLP). MaxEnt uses presence/background directly.
Grid search picks the combination with the highest mean CV ROC-AUC (lower SD breaks ties), and the tuned version is kept
only if it beats the default.

In [ ]:
def build(name, params, y_train):
    if name == "Random_Forest":
        return RandomForestClassifier(class_weight="balanced", random_state=SEED, n_jobs=-1, **params)
    if name == "XGBoost":
        return XGBClassifier(objective="binary:logistic", eval_metric="logloss", tree_method="hist",
                             scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
                             random_state=SEED, n_jobs=-1, **params)
    if name == "SVM":   # Platt scaling for probabilities
        return Pipeline([("scale", StandardScaler()),
                         ("model", CalibratedClassifierCV(SVC(class_weight="balanced", random_state=SEED, **params),
                                                          method="sigmoid", cv=5, ensemble=False))])
    if name == "MaxEnt":
        return ela.MaxentModel(random_state=SEED, use_sklearn=True, **params)
    if name == "MLP":
        return Pipeline([("scale", StandardScaler()),
                         ("model", MLPClassifier(random_state=SEED, max_iter=2000, early_stopping=True,
                                                 validation_fraction=0.15, n_iter_no_change=25, **params))])


def predict(name, model, X):
    if name == "MaxEnt":
        return np.asarray(model.predict(X), float).reshape(-1)
    return model.predict_proba(X)[:, 1]


def fit(name, params, X_train, y_train):
    X_train, y_train = X_train.reset_index(drop=True), pd.Series(np.asarray(y_train))
    if name == "MLP":   # oversample presences to the same number as pseudo-absences
        pos = np.where(y_train == 1)[0]
        extra = np.random.default_rng(SEED).choice(pos, (y_train == 0).sum() - len(pos), replace=True)
        X_train = pd.concat([X_train, X_train.iloc[extra]], ignore_index=True)
        y_train = pd.concat([y_train, y_train.iloc[extra]], ignore_index=True)
    return build(name, params, y_train).fit(X_train, y_train)


def fold_splits():
    for f in sorted(folds.unique()):
        yield f, np.where(folds != f)[0], np.where(folds == f)[0]


def spatial_cv(name, params, features=None):
    """5-fold spatial CV -> out-of-fold predictions, fold models and fold AUCs."""
    X = X_dev[features or FEATURES]
    oof, models, aucs = np.full(len(X), np.nan), [], []
    for f, tr, te in fold_splits():
        m = fit(name, params, X.iloc[tr], y_dev.iloc[tr])
        oof[te] = predict(name, m, X.iloc[te])
        aucs.append(roc_auc_score(y_dev.iloc[te], oof[te]))
        models.append(m)
    return {"oof": oof, "models": models, "mean_auc": np.mean(aucs), "sd_auc": np.std(aucs, ddof=1)}

In [ ]:
GRIDS = {
    "Random_Forest": {"n_estimators": [200, 500, 800], "max_depth": [None, 8, 16], "min_samples_split": [2, 5],
                      "min_samples_leaf": [1, 2], "max_features": ["sqrt", 0.5]},
    "XGBoost": {"n_estimators": [200, 400], "max_depth": [3, 6, 9], "learning_rate": [0.03, 0.10],
                "subsample": [0.75, 1.0], "colsample_bytree": [0.75, 1.0], "min_child_weight": [1, 5]},
    "SVM": {"C": [0.1, 1, 10, 100], "gamma": ["scale", 0.001, 0.01, 0.1], "kernel": ["rbf"]},
    "MaxEnt": {"beta_multiplier": [0.5, 1.0, 1.5, 2.0],
               "feature_types": [["linear", "quadratic", "hinge"], ["linear", "quadratic", "product", "hinge"]]},
    "MLP": {"hidden_layer_sizes": [(8,), (16,), (32,), (64,), (64, 32), (128, 64)], "activation": ["relu", "tanh"],
            "alpha": [0.0001, 0.001, 0.01], "learning_rate_init": [0.0005, 0.001]},
}
DEFAULTS = {"Random_Forest": {}, "XGBoost": {}, "SVM": {}, "MaxEnt": {},
            "MLP": {"hidden_layer_sizes": (64, 32), "activation": "relu", "alpha": 0.001, "learning_rate_init": 0.001}}

# this takes a long time (hundreds of fits per model), so results are saved
selected = {}
for name in MODEL_NAMES:
    search = []
    for params in ParameterGrid(GRIDS[name]):
        r = spatial_cv(name, params)
        search.append({"params": params, "mean_auc": r["mean_auc"], "sd_auc": r["sd_auc"]})
    search = sorted(search, key=lambda s: (-s["mean_auc"], s["sd_auc"]))
    tuned = spatial_cv(name, search[0]["params"])
    default = spatial_cv(name, DEFAULTS[name])
    use_tuned = (tuned["mean_auc"], -tuned["sd_auc"]) > (default["mean_auc"], -default["sd_auc"])
    selected[name] = {**(tuned if use_tuned else default),
                      "params": search[0]["params"] if use_tuned else DEFAULTS[name],
                      "version": "tuned" if use_tuned else "default"}
    print(f"{name:14s} {selected[name]['version']:8s} CV AUC {selected[name]['mean_auc']:.3f} ± {selected[name]['sd_auc']:.3f}")

json.dump({m: {"version": s["version"], "params": s["params"]} for m, s in selected.items()},
          open(OUT_DIR / "selected_hyperparameters.json", "w"), indent=2, default=str)

## 6. Ensemble and evaluation

Ensemble weight = mean CV AUC − 0.5 (set to 0 if not better than random), normalised to sum to 1.
For the holdout, each algorithm is the average of its 5 fold models. The Youden threshold for the holdout comes from the
pooled out-of-fold predictions, so it is never set using the holdout itself. 95% CIs are BCa bootstraps (2000 resamples,
presences and pseudo-absences resampled separately), and models are compared with DeLong's test.

In [ ]:
raw_w = {m: max(selected[m]["mean_auc"] - 0.5, 0) for m in MODEL_NAMES}
weights = {m: w / sum(raw_w.values()) for m, w in raw_w.items()}
print({m: round(w, 3) for m, w in weights.items()})


def holdout_pred(name, models, X):
    return np.mean([predict(name, m, X) for m in models], axis=0)

probs_hold = {m: holdout_pred(m, selected[m]["models"], X_hold[FEATURES]) for m in MODEL_NAMES}
probs_hold["Ensemble"] = sum(weights[m] * probs_hold[m] for m in MODEL_NAMES)
oof = {m: selected[m]["oof"] for m in MODEL_NAMES}
oof["Ensemble"] = sum(weights[m] * oof[m] for m in MODEL_NAMES)

In [ ]:
def youden(y_true, p):
    fpr, tpr, thr = roc_curve(y_true, p)
    t = thr[np.argmax(tpr - fpr)]
    return float(t) if np.isfinite(t) else 0.5


def metrics(y_true, p, thr):
    y_true = np.asarray(y_true).astype(int)
    pred = (np.asarray(p) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    sens, spec = tp / (tp + fn), tn / (tn + fp)
    return {"AUC": roc_auc_score(y_true, p), "PR_AUC": average_precision_score(y_true, p),
            "sensitivity": sens, "specificity": spec, "accuracy": (tp + tn) / len(y_true),
            "F1": f1_score(y_true, pred, zero_division=0), "PPV": tp / (tp + fp) if tp + fp else np.nan,
            "NPV": tn / (tn + fn) if tn + fn else np.nan, "kappa": cohen_kappa_score(y_true, pred),
            "TSS": sens + spec - 1, "TP": tp, "TN": tn, "FP": fp, "FN": fn}


def bca_ci(y_true, p, thr, metric, n_boot=N_BOOTSTRAP, alpha=0.05, seed=SEED):
    """Bias-corrected and accelerated bootstrap CI, resampling each class separately."""
    y_true, p = np.asarray(y_true).astype(int), np.asarray(p)
    i1, i0 = np.where(y_true == 1)[0], np.where(y_true == 0)[0]
    rng = np.random.default_rng(seed)
    theta = metrics(y_true, p, thr)[metric]
    boot = []
    for _ in range(n_boot):
        idx = np.r_[rng.choice(i1, len(i1)), rng.choice(i0, len(i0))]
        boot.append(metrics(y_true[idx], p[idx], thr)[metric])
    boot = np.array(boot, float)
    boot = boot[np.isfinite(boot)]
    z0 = stats.norm.ppf(np.clip(np.mean(boot < theta), 1e-6, 1 - 1e-6))
    jack = np.array([metrics(np.delete(y_true, i), np.delete(p, i), thr)[metric] for i in range(len(y_true))])
    jack = jack[np.isfinite(jack)]
    d = jack.mean() - jack
    a = (d ** 3).sum() / (6 * ((d ** 2).sum() ** 1.5)) if (d ** 2).sum() > 0 else 0.0
    zl, zu = stats.norm.ppf(alpha / 2), stats.norm.ppf(1 - alpha / 2)
    lo = stats.norm.cdf(z0 + (z0 + zl) / (1 - a * (z0 + zl)))
    hi = stats.norm.cdf(z0 + (z0 + zu) / (1 - a * (z0 + zu)))
    return np.percentile(boot, 100 * lo), np.percentile(boot, 100 * hi)

In [ ]:
results = []
for name, p in probs_hold.items():
    thr = youden(y_dev, oof[name])                     # threshold from pooled OOF predictions
    for label, t in [("default_0.5", 0.5), ("youden", thr)]:
        row = {"model": name, "threshold_type": label, "threshold": t, **metrics(y_hold, p, t)}
        for m in ["AUC", "PR_AUC", "kappa", "TSS"]:
            row[f"{m}_CI"] = bca_ci(y_hold, p, t, m)
        results.append(row)
holdout_results = pd.DataFrame(results)
holdout_results.to_csv(OUT_DIR / "holdout_results.csv", index=False)
holdout_results[holdout_results.threshold_type == "youden"][["model", "threshold", "AUC", "AUC_CI", "PR_AUC", "TSS", "TP", "FN", "FP"]].round(3)

In [ ]:
# DeLong test for correlated ROC curves (fast version from Sun & Xu, 2014)
def midrank(x):
    order = np.argsort(x); z = x[order]; n = len(x); t = np.zeros(n)
    i = 0
    while i < n:
        j = i
        while j < n and z[j] == z[i]:
            j += 1
        t[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    out = np.empty(n); out[order] = t
    return out


def delong(y_true, p_a, p_b):
    y_true = np.asarray(y_true)
    order = np.argsort(-y_true, kind="stable")
    m = int(y_true.sum())
    preds = np.vstack([np.asarray(p_a)[order], np.asarray(p_b)[order]])
    n = preds.shape[1] - m
    tx = np.array([midrank(r[:m]) for r in preds]); ty = np.array([midrank(r[m:]) for r in preds])
    tz = np.array([midrank(r) for r in preds])
    aucs = tz[:, :m].sum(1) / m / n - (m + 1) / (2 * n)
    v01, v10 = (tz[:, :m] - tx) / n, 1 - (tz[:, m:] - ty) / m
    cov = np.cov(v01) / m + np.cov(v10) / n
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    z = (aucs[0] - aucs[1]) / np.sqrt(var) if var > 0 else np.nan
    return aucs[0], aucs[1], (2 * (1 - stats.norm.cdf(abs(z))) if np.isfinite(z) else 1.0)

delong_table = pd.DataFrame([(a, b, *delong(y_hold, probs_hold[a], probs_hold[b]))
                             for a, b in combinations(probs_hold, 2)], columns=["model_a", "model_b", "AUC_a", "AUC_b", "p"])
delong_table.round(4)

## 7. Permutation importance and SHAP

Both are calculated on each fold's spatially held-out test data. Permutation importance = mean drop in ROC-AUC when a
predictor is shuffled (5 times). SHAP uses the permutation explainer (minimum number of evaluations) on up to 100 test
points per fold, with 50 training points as background.

In [ ]:
rng = np.random.default_rng(SEED)
perm_rows = []
for name in MODEL_NAMES:
    for f, tr, te in fold_splits():
        model = selected[name]["models"][f]
        X_te, y_te = X_dev.iloc[te].reset_index(drop=True), y_dev.iloc[te]
        base = roc_auc_score(y_te, predict(name, model, X_te))
        for feat in FEATURES:
            drops = []
            for _ in range(PERM_REPEATS):
                X_p = X_te.copy()
                X_p[feat] = rng.permutation(X_p[feat].values)
                drops.append(base - roc_auc_score(y_te, predict(name, model, X_p)))
            perm_rows.append({"model": name, "fold": f, "feature": feat, "auc_drop": np.mean(drops)})
perm_imp = pd.DataFrame(perm_rows).groupby(["model", "feature"]).auc_drop.mean().reset_index()
perm_imp.sort_values(["model", "auc_drop"], ascending=[True, False]).groupby("model").head(5)

In [ ]:
def ensemble_fn(fold):
    def f(arr):
        X = pd.DataFrame(arr, columns=FEATURES)
        return sum(weights[m] * predict(m, selected[m]["models"][fold], X) for m in MODEL_NAMES)
    return f

shap_values, shap_X = [], []
for f, tr, te in fold_splits():
    X_te = X_dev.iloc[te]
    sample = X_te.sample(min(SHAP_SAMPLES, len(X_te)), random_state=SEED + f)
    background = X_dev.iloc[tr].sample(SHAP_BACKGROUND, random_state=SEED + 100 + f)
    explainer = shap.Explainer(ensemble_fn(f), background, algorithm="permutation", feature_names=FEATURES)
    ex = explainer(sample, max_evals=max(2 * len(FEATURES) + 1, 11))
    shap_values.append(ex.values); shap_X.append(sample)

shap_values = np.vstack(shap_values); shap_X = pd.concat(shap_X)
shap.summary_plot(shap_values, shap_X, max_display=15, show=False)
plt.title("Ensemble SHAP (spatial test folds)")
plt.savefig(OUT_DIR / "shap_ensemble_beeswarm.png", dpi=300, bbox_inches="tight")
plt.show()
# the same loop was run for each single algorithm by swapping ensemble_fn for that model's predict

## 8. Prediction map

Every marine cell is predicted with the fold models (averaged per algorithm) and the ensemble weights. The models are not
refitted with the holdout, and no calibration is applied, so values are **relative suitability**.

In [ ]:
X_all = transform_table(grid, FEATURES)
ok = X_all.notna().all(axis=1).to_numpy()
surface = np.full(shape, np.nan, dtype="float32")
ens = np.zeros(ok.sum())
for m in MODEL_NAMES:
    ens += weights[m] * holdout_pred(m, selected[m]["models"], X_all[ok])
surface[grid.row.values[ok], grid.col.values[ok]] = ens

profile = dict(driver="GTiff", height=shape[0], width=shape[1], count=1, dtype="float32",
               crs=crs, transform=transform, nodata=np.nan, compress="deflate")
with rasterio.open(OUT_DIR / f"{EXTENT}_{VARIANT}_Ensemble.tif", "w", **profile) as dst:
    dst.write(surface, 1)

thr_ens = youden(y_dev, oof["Ensemble"])
fig, ax = plt.subplots(figsize=(8, 9))
extent = [transform.c, transform.c + shape[1] * transform.a, transform.f + shape[0] * transform.e, transform.f]
im = ax.imshow(surface, extent=extent, cmap="viridis", vmin=0, vmax=1)
ax.scatter(hold.lon[hold.presence == 1], hold.lat[hold.presence == 1], s=15, c="red", edgecolor="k", label="holdout presence")
plt.colorbar(im, label="Relative suitability")
ax.set_title(f"Ensemble, {EXTENT} (Youden threshold {thr_ens:.3f})")
ax.legend()
plt.show()

## 9. Compact model: backward elimination

Starting from the all-predictor set, one predictor is removed per step, hardest-to-get first:
tier 3 = Copernicus variables at surface and seabed and seabed kinetic energy, tier 2 = single-depth Copernicus products and
AIS vessel/route density, tier 1 = bathymetry and derivatives, port distances and substrate.
Within a tier the least important (ensemble permutation importance) goes first. A removal is accepted if the new development
ensemble is **not significantly worse than the original full model** (DeLong, p ≥ 0.05). Up to 5 candidates are tried per step,
hyperparameters stay fixed, and it stops at 3 predictors or when nothing more can be removed. The holdout is not used here.

In [ ]:
def tier(name):
    if name in {"bathymetry", "bathymetry_sd", "slope", "slope_sd", "vrm"} or name.startswith(("substrate_", "distance_port")):
        return 1
    if name == "kinetic_energy_seabed":
        return 3
    if name.startswith(("current_u_", "current_v_", "current_speed_", "salinity_", "temperature_")) and \
            name.endswith(("_surface", "_seabed")):
        return 3
    return 2   # chlorophyll, waves, wind, SSH, Stokes drift, vessel and route density


def ensemble_cv(features):
    res = {m: spatial_cv(m, selected[m]["params"], features) for m in MODEL_NAMES}
    w = {m: max(res[m]["mean_auc"] - 0.5, 0) for m in MODEL_NAMES}
    w = {m: v / sum(w.values()) for m, v in w.items()}
    return res, w, sum(w[m] * res[m]["oof"] for m in MODEL_NAMES)


def ensemble_importance(features, res, w):
    out = {feat: [] for feat in features}
    for f, tr, te in fold_splits():
        X_te, y_te = X_dev[features].iloc[te].reset_index(drop=True), y_dev.iloc[te]
        ens_pred = lambda X: sum(w[m] * predict(m, res[m]["models"][f], X) for m in MODEL_NAMES)
        base = roc_auc_score(y_te, ens_pred(X_te))
        for feat in features:
            for _ in range(PERM_REPEATS):
                X_p = X_te.copy(); X_p[feat] = rng.permutation(X_p[feat].values)
                out[feat].append(base - roc_auc_score(y_te, ens_pred(X_p)))
    return {k: np.mean(v) for k, v in out.items()}


full_oof = oof["Ensemble"]          # the original all-predictor ensemble, kept as the reference
current = list(FEATURES)
res, w, _ = ensemble_cv(current)
while len(current) > 3:
    imp = ensemble_importance(current, res, w)
    order = sorted(current, key=lambda f: (-tier(f), imp[f]))[:5]
    for feat in order:
        trial = [f for f in current if f != feat]
        res_t, w_t, ens_t = ensemble_cv(trial)
        p = delong(y_dev, full_oof, ens_t)[2]
        print(f"  try removing {feat} (tier {tier(feat)}): DeLong p = {p:.3f}")
        if p >= 0.05:
            current, res, w = trial, res_t, w_t
            print(f"  -> removed, {len(current)} left")
            break
    else:
        print("No candidate could be removed, stopping.")
        break

print("Compact predictor set:", current)

In [ ]:
# compare the compact ensemble with the full one on the holdout (once, at the end)
compact_hold = sum(w[m] * holdout_pred(m, res[m]["models"], X_hold[current]) for m in MODEL_NAMES)
auc_full, auc_compact, p = delong(y_hold, probs_hold["Ensemble"], compact_hold)
print(f"Holdout AUC full {auc_full:.3f} vs compact {auc_compact:.3f} (DeLong p = {p:.3f})")